# Assignment 2 - Interactive Isosurface & Histogram

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import vtk
from vtk.util.numpy_support import vtk_to_numpy

In [2]:
DATA_FILE=Path("mixture.vti")
INITIAL_ISO=0.0
WINDOW=0.25
NBINS_FULL=50
NBINS_SUBSET=30

In [3]:
reader=vtk.vtkXMLImageDataReader()
reader.SetFileName(str(DATA_FILE))
reader.Update()
img=reader.GetOutput()

In [4]:
values=vtk_to_numpy(img.GetPointData().GetScalars()).astype(np.float32)

dims=img.GetDimensions()
spacing=img.GetSpacing()
origin=img.GetOrigin()

x=np.arange(dims[0])*spacing[0]+origin[0]
y=np.arange(dims[1])*spacing[1]+origin[1]
z=np.arange(dims[2])*spacing[2]+origin[2]
X,Y,Z=np.meshgrid(x,y,z,indexing="ij")
X=X.ravel().tolist(); Y=Y.ravel().tolist(); Z=Z.ravel().tolist()
VALUES=values.tolist()
MIN_VAL=float(values.min()); MAX_VAL=float(values.max())

In [5]:
def plasma_color(v):
    t=(v-MIN_VAL)/(MAX_VAL-MIN_VAL)
    t=max(0,min(1,t))
    c=px.colors.sample_colorscale("Plasma",[t])[0]
    return [[0,c],[1,c]]

In [6]:
def hist_data(iso):
    if abs(iso-INITIAL_ISO)<1e-12:
        return VALUES,[MIN_VAL,MAX_VAL],NBINS_FULL
    m=(values>=iso-WINDOW)&(values<=iso+WINDOW)
    return values[m].tolist(),[iso-WINDOW,iso+WINDOW],NBINS_SUBSET

In [7]:
iso=go.Isosurface(
    x=X,y=Y,z=Z,value=VALUES,
    isomin=INITIAL_ISO,isomax=INITIAL_ISO,
    surface_count=1,
    colorscale=plasma_color(INITIAL_ISO),
    cmin=MIN_VAL,cmax=MAX_VAL,
    showscale=False,
    caps=dict(x_show=False,y_show=False,z_show=False)
)

hist=go.Histogram(x=VALUES,nbinsx=NBINS_FULL,marker_color="royalblue")

fig=go.FigureWidget(make_subplots(rows=1,cols=2,
                                  specs=[[{"type":"scene"},{"type":"xy"}]],
                                  column_widths=[0.55,0.45]))
fig.add_trace(iso,1,1)
fig.add_trace(hist,1,2)
fig.update_layout(width=1100,height=550,showlegend=False,
                  margin=dict(l=10,r=10,t=30,b=10))
fig.update_xaxes(title="Vortex Scalar Value",row=1,col=2)
fig.update_yaxes(title="Frequency",row=1,col=2)

slider=widgets.FloatSlider(
    value=INITIAL_ISO,min=MIN_VAL,max=MAX_VAL,
    step=(MAX_VAL-MIN_VAL)/250,
    description="Isovalue",
    continuous_update=False,
    readout=True,
    readout_format=".2f",
    layout=widgets.Layout(width="500px")
)
reset=widgets.Button(description="Reset",button_style="warning")

def update_visualization(iso):
    h, xr, bins=hist_data(iso)
    with fig.batch_update():
        fig.data[0].isomin=iso
        fig.data[0].isomax=iso
        fig.data[0].colorscale=plasma_color(iso)
        fig.data[1].x=h
        fig.data[1].nbinsx=bins
        fig.layout.xaxis.range=xr

def slider_changed(change):
    update_visualization(change["new"])

def reset_clicked(_):
    slider.value=INITIAL_ISO
    update_visualization(INITIAL_ISO)

slider.observe(slider_changed,names="value")
reset.on_click(reset_clicked)

display(widgets.HBox([slider,reset]))
display(fig)


FigureWidget({
    'data': [{'caps': {'x': {'show': False}, 'y': {'show': False}, 'z': {'show': False}},
              'cmax': 0.43280163407325745,
              'cmin': -0.9935540556907654,
              'colorscale': [[0, 'rgb(241, 131, 76)'], [1, 'rgb(241, 131, 76)']],
              'isomax': 0.0,
              'isomin': 0.0,
              'scene': 'scene',
              'showscale': False,
              'surface': {'count': 1},
              'type': 'isosurface',
              'uid': '40353556-eea5-477a-a2db-13fb321cf1b1',
              'value': [-0.04087147116661072, -0.04280706122517586,
                        -0.05014687404036522, ..., -0.7243356108665466,
                        -0.7158576250076294, -0.7285129427909851],
              'x': [7.449999999664669e-05, 7.449999999664669e-05,
                    7.449999999664669e-05, ..., 148.99992550000002,
                    148.99992550000002, 148.99992550000002],
              'y': [7.449999999664669e-05, 7.449999999664669e-05,